# House Price Analysis & Prediction

**Slab 1 – For Beginners**

This notebook is designed as a complete, reproducible submission. Run the cells from top to bottom.

## 1. Objective
Analyze housing data and predict house prices. The workflow covers missing values, outliers, categorical encoding, feature selection, regression models, MAE/RMSE/R² and feature importance.

**Dataset:** Ames Housing / House Prices data with `SalePrice` as the target.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

sns.set_theme(style="whitegrid")

# OpenML provides the Ames House Prices dataset.
house = fetch_openml(name="house_prices", as_frame=True, parser="auto")
df = house.frame.copy()

print("Shape:", df.shape)
display(df.head())

## 2. Cleaning and target analysis

In [ ]:
# Standardize the target name and inspect quality.
df.columns = [str(c) for c in df.columns]
target = "SalePrice"

print("Missing values in top columns:")
display(df.isna().sum().sort_values(ascending=False).head(20))

print("Duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates().copy()

df[target] = pd.to_numeric(df[target], errors="coerce")
df = df.dropna(subset=[target]).copy()

display(df[target].describe().to_frame().T.round(2))

plt.figure(figsize=(9,5))
sns.histplot(df[target], kde=True)
plt.title("House Price Distribution")
plt.xlabel("SalePrice")
plt.show()

plt.figure(figsize=(8,5))
sns.boxplot(x=df[target])
plt.title("SalePrice Outlier View")
plt.show()

## 3. Exploratory analysis and feature relationships

In [ ]:
# Strong numerical relationships with price.
numeric = df.select_dtypes(include=np.number)
corr = numeric.corr(numeric_only=True)[target].sort_values(ascending=False)
display(corr.head(15).to_frame("Correlation with SalePrice"))

top_corr = corr.drop(target).abs().sort_values(ascending=False).head(10).index
plt.figure(figsize=(10,7))
sns.heatmap(df[list(top_corr)+[target]].corr(), cmap="coolwarm", center=0, annot=False)
plt.title("Heatmap of Strongest Numeric Relationships")
plt.show()

if "OverallQual" in df.columns:
    plt.figure(figsize=(9,5))
    sns.boxplot(data=df, x="OverallQual", y=target)
    plt.title("Overall Quality vs Sale Price")
    plt.show()

if "GrLivArea" in df.columns:
    plt.figure(figsize=(9,5))
    sns.scatterplot(data=df, x="GrLivArea", y=target, alpha=0.6)
    plt.title("Living Area vs Sale Price")
    plt.show()

## 4. Outlier handling and feature selection
For a beginner-friendly regression pipeline, we avoid manually deleting every unusual observation. Instead, we remove only clearly invalid records (for example, non-positive prices or impossible living areas where applicable), then let the models handle remaining variation.

We also exclude the identifier `Id` because it is not a meaningful property characteristic.

In [ ]:
df = df[df[target] > 0].copy()

if "GrLivArea" in df.columns:
    df = df[df["GrLivArea"] > 0].copy()

X = df.drop(columns=[target])
if "Id" in X.columns:
    X = X.drop(columns=["Id"])
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=np.number).columns.tolist()

preprocessor = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), numeric_features),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_features)
])

models = {
    "Ridge Regression": Ridge(alpha=10.0),
    "Random Forest": RandomForestRegressor(
        n_estimators=300, random_state=42, n_jobs=-1, max_features=0.8
    ),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42)
}

results = []
fitted = {}

for name, model in models.items():
    pipe = Pipeline([("prep", preprocessor), ("model", model)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": mean_squared_error(y_test, pred, squared=False),
        "R2": r2_score(y_test, pred)
    })
    fitted[name] = (pipe, pred)

results_df = pd.DataFrame(results).sort_values("RMSE")
display(results_df.round(2))

## 5. Best model evaluation and feature importance

In [ ]:
best_name = results_df.iloc[0]["Model"]
best_pipe, best_pred = fitted[best_name]
print("Best model:", best_name)

plt.figure(figsize=(7,7))
sns.scatterplot(x=y_test, y=best_pred, alpha=0.65)
lims = [min(y_test.min(), best_pred.min()), max(y_test.max(), best_pred.max())]
plt.plot(lims, lims, "--")
plt.xlabel("Actual SalePrice")
plt.ylabel("Predicted SalePrice")
plt.title(f"Actual vs Predicted House Prices – {best_name}")
plt.show()

# Feature importance for tree models.
if best_name in ["Random Forest", "Gradient Boosting"]:
    feature_names = best_pipe.named_steps["prep"].get_feature_names_out()
    importance = pd.Series(
        best_pipe.named_steps["model"].feature_importances_,
        index=feature_names
    ).sort_values(ascending=False).head(20)

    display(importance.to_frame("Importance"))
    importance.sort_values().plot(kind="barh", figsize=(10,7), title="Top House Price Features")
    plt.tight_layout()
    plt.show()

## 6. Final observations
Use the executed results to report:

- The model with the lowest MAE and RMSE.
- The model with the highest R².
- The strongest property characteristics associated with price.
- Whether overall quality, living area, garage/basement characteristics or neighborhood variables are important.
- How missing categorical/numerical values were handled.
- How outliers were inspected without blindly deleting legitimate expensive homes.
- Business implications for buyers, sellers and property valuation.

The final report should emphasize that **correlation or model importance does not automatically prove causation**.